# Adım 2 — Kafka StreamingBu notebook, **Kafka Producer** yapısını, mesaj formatını ve veri akışı simülasyonunu belgeler.## Mimari```  Edge-IIoTset CSV          Kafka Broker             Spark Consumer  ┌──────────────┐    ┌─────────────────────┐    ┌──────────────────┐  │ ~157K satır   │───▶│ topic:               │───▶│ readStream       │  │ ML-EdgeIIoT-  │    │ iot-network-traffic  │    │  .format("kafka")│  │ dataset.csv   │    │ partitions: 3        │    │  .subscribe(...)│  └──────────────┘    └─────────────────────┘    └──────────────────┘       producer.py          :9092 / :29092         run_streaming_pipeline.py```## Producer Özellikleri- CSV satırlarını JSON mesajlara dönüştürür- Her mesaja `timestamp`, `source_ip`, `attack_type`, `flow_id` ekler- Ayarlanabilir hız: `--rate 100` (msg/s), `--no-delay` (tam hız), `--max-messages 5000`- Kafka bağlantısı için retry mekanizması (max 30 deneme, 5s aralık)

In [ ]:
# Producer kodunu oku ve gösterimport osproducer_path = os.path.join(os.pardir, "kafka", "producer.py")with open(producer_path) as f:    code_text = f.read()print(f"Dosya: kafka/producer.py  |  Satır sayısı: {len(code_text.splitlines())}")print("═" * 60)print(code_text)

## Kafka Mesaj Formatı (JSON)Producer'ın Kafka'ya gönderdiği her mesajın yapısı:```json{  "timestamp":    "2023-06-15 14:32:01",  "source_ip":    "192.168.1.105",  "dest_ip":      "10.0.0.1",  "attack_type":  "DDoS_UDP",  "flow_id":      "192.168.1.105-10.0.0.1-443-52341-6",  "tcp_srcport":  52341,  "tcp_dstport":  443,  "tcp_seq":      1548762,  "tcp_ack":      0,  "tcp_len":      1460,  "tcp_flags":    "0x0002",  "...":          "diğer 50+ kolon"}```## Konfigürasyon Parametreleri| Parametre | Varsayılan | Açıklama ||-----------|-----------|----------|| `--rate` | 50 | Saniyedeki mesaj sayısı || `--no-delay` | False | Tam hız (delay olmadan gönder) || `--max-messages` | 0 | Maksimum mesaj sayısı (0 = sınırsız) || `--csv-path` | `/app/data/raw/*.csv` | Kaynak CSV dosyası || `--topic` | `iot-network-traffic` | Kafka topic adı || `--bootstrap-servers` | `kafka:9092` | Kafka broker adresi |

In [ ]:
# Örnek bir Kafka mesajı simüle etimport jsonfrom datetime import datetimesample_msg = {    "timestamp": datetime.now().isoformat(),    "source_ip": "192.168.1.105",    "dest_ip": "10.0.0.1",    "attack_type": "DDoS_UDP",    "flow_id": "192.168.1.105-10.0.0.1-443-52341-6",    "tcp_srcport": 52341,    "tcp_dstport": 443,    "tcp_seq": 1548762,    "tcp_ack": 0,    "tcp_len": 1460,    "tcp_flags": "0x0002",    "frame_len": 1514,    "Attack_label": 1,}print("Örnek Kafka mesajı (JSON):")print("─" * 50)print(json.dumps(sample_msg, indent=2, default=str))print(f"\nMesaj boyutu: ~{len(json.dumps(sample_msg))} byte")

## Çalıştırma Komutları```bash# 1) Kafka topic oluşturdocker exec -it kafka kafka-topics.sh \  --create --topic iot-network-traffic \  --partitions 3 --replication-factor 1 \  --bootstrap-server localhost:9092# 2) Topic'i doğruladocker exec -it kafka kafka-topics.sh \  --list --bootstrap-server localhost:9092# 3) Producer'ı başlatdocker compose up --build kafka-producer# 4) Mesajları izle (opsiyonel)docker exec -it kafka kafka-console-consumer.sh \  --topic iot-network-traffic \  --bootstrap-server localhost:9092 \  --from-beginning --max-messages 5```